# 02 — Scheduling benchmark: sensitivity of the Double DQN agent (notebook-scale)

This notebook retrains the agent with **one setting changed at a time** and evaluates every
variant on the same paired workloads, with the same small training budget and the same seeds:

1. the reward used here (response time is a penalty) versus an alternative sign convention for
   the response-time term, and two reward-weight variants;
2. the three load levels (light, medium, heavy) for the default agent and the baselines;
3. one hyperparameter, the learning rate, at three values.

The point is **sensitivity**, not a best result: how much the agent's scheduling metrics move
when one assumption changes, compared with the seed-to-seed spread.

## What this notebook does and does not show

**It shows**

- how strongly the agent's results depend on the reward definition, the load level and the
  learning rate, each variant trained with the same reduced budget on the same seeds;
- the seed-to-seed spread of every variant (per-seed values are drawn as dots), so that a
  difference can be judged against it;
- the per-workload distribution under each load level, because averages hide tail failures.

**It does not show**

- a tuned or "best" agent. The default settings were fixed on development seeds before the
  reference run and are not changed by anything here. Picking the variant that looks best in
  this notebook would be tuning on the reporting seeds. A variant that looks better here is a
  sign of sensitivity, not a recommended setting;
- the reference ablations. Those use the full training budget and five seeds and are in
  `experiments/results/RESULTS.md` (section 3, Ablations). Because this notebook
  uses a smaller training budget, its "default" agent is not the reference agent either;
- anything outside the simulator: single CPU, synthetic workloads, every setting
  `[Assumption — not in paper]`.

The training return of different reward variants is **not comparable** (each variant optimises
a different reward), so only scheduling metrics are compared below.

## Parameters

Every knob of this notebook is in the next cell.

In [ ]:
import os
from pathlib import Path

SCHEDULER_CONFIG = "configs/scheduler_ddqn.yaml"
BASELINES_CONFIG = "configs/scheduler_baselines.yaml"
BEFORE_AFTER_CONFIG = "configs/before_after.yaml"

# Applied to EVERY variant, including the default. Scale choice: half of the reference
# training budget (300 episodes) so that 6 variants x 5 seeds finish in a few minutes on a
# laptop CPU with a handful of worker processes. The exploration schedule is unchanged (it is counted in environment steps and
# completes well within this budget). The smaller budget applies equally to every variant,
# so it changes the level of every result, not the comparison between variants.
BASE_OVERRIDES = ["training.episodes=150"]

# All five reporting seeds, the same for every variant (paired design). Fewer seeds would be
# faster but not neutral: the agent's catastrophic heavy-load workloads occur on some seeds
# and not on others, so a seed subset can hide (or exaggerate) exactly the tail behaviour this
# notebook is meant to show. Reduce only for a quick look.
SEEDS = [0, 1, 2, 3, 4]

# Variants are named ablations from the `ablations` block of SCHEDULER_CONFIG; each is a set
# of dotted overrides applied on top of the config (one setting changed at a time).
REWARD_VARIANTS = ["reward_paper_sign_as_printed", "reward_turnaround_only", "reward_responsive"]
LEARNING_RATE_VARIANTS = ["lr_low", "lr_high"]   # with the default: three learning rates

# Worker processes (one training run per process). Results do not depend on this number.
WORKERS = max(1, min(6, os.cpu_count() or 1))

OUTPUT_DIR = Path("experiments/results/runs/notebooks/02_scheduler_ablation")

In [ ]:
import os
from pathlib import Path

# The runner (python -m experiments.run_notebooks) starts the kernel in the repository root.
# If you opened the notebook from the notebooks/ folder instead, move up one level.
if not Path("configs").is_dir() and Path("../configs").is_dir():
    os.chdir("..")
assert Path("configs").is_dir(), "Run this notebook with the repository root as working directory."

from IPython.display import Image, Markdown, display


def provenance(config_paths, overrides, seeds, extra=None):
    """Print where every number below comes from: config file(s), overrides and seeds."""
    paths = [config_paths] if isinstance(config_paths, (str, Path)) else list(config_paths)
    print("config:    " + ", ".join(str(path) for path in paths))
    print("overrides: " + (", ".join(overrides) if overrides else "none"))
    print(f"seeds:     {list(seeds)} ({len(seeds)} seed(s); notebook-scale run, not the reference run)")
    if extra:
        print(extra)


def show_table(header, body, caption=""):
    """Render a small Markdown table (cells are already formatted strings)."""
    lines = ["| " + " | ".join(header) + " |", "|" + "|".join(["---"] * len(header)) + "|"]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in body]
    display(Markdown((caption + "\n\n" if caption else "") + "\n".join(lines)))

In [ ]:
%matplotlib inline
import math
import multiprocessing
import time

import matplotlib.pyplot as plt
import numpy as np

from experiments.build_results import MetricCfg, find_row, measurement
from experiments.run_scheduler_benchmark import DDQN_METHOD, run_seed, summarise
from src.common import plotting
from src.common.before_after import make_row, row_cells
from src.common.config import load_config
from src.common.io import write_csv
from src.common.labels import REPRODUCED
from src.common.metrics import format_mean_std, summarise_by_seed
from src.common.plotting import bar_panel
from src.scheduler import RAW_EPISODE_FIELDS
from src.scheduler.config import apply_ablation, load_baselines_config, scheduler_config_from_dict

plotting.apply_style()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = load_config(SCHEDULER_CONFIG, overrides=BASE_OVERRIDES)
bcfg = load_baselines_config(BASELINES_CONFIG)
VARIANTS = ["default", *REWARD_VARIANTS, *LEARNING_RATE_VARIANTS]
configs = {"default": scheduler_config_from_dict(raw)}
changes = {"default": "nothing (config + base overrides)"}
for name in VARIANTS[1:]:
    configs[name] = scheduler_config_from_dict(apply_ablation(raw, name))
    changes[name] = "; ".join(f"`{k}` = `{v}`" for k, v in raw["ablations"][name].items())

TRACK = load_config(BEFORE_AFTER_CONFIG)["scheduling"]
BY_KEY = {entry["key"]: MetricCfg(**entry) for entry in TRACK["metrics"]}
LABELS = TRACK["method_labels"]
BEFORE, AFTER, REFERENCES = list(TRACK["before_methods"]), TRACK["after_method"], list(TRACK["reference_methods"])
ORDER = BEFORE + [AFTER] + REFERENCES
ROLE = {**{m: "before" for m in BEFORE}, AFTER: "after", **{m: "reference" for m in REFERENCES}}
RUN_LABEL = f"notebook-scale run, {len(SEEDS)} seeds, {configs['default'].training.episodes} training episodes"

provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], BASE_OVERRIDES, SEEDS)
show_table(
    ["Variant", "What changed", "learning rate", "alternative reward sign", "w_wait", "w_response", "w_slo_violation"],
    [[name, changes[name], c.agent.learning_rate, c.reward.paper_sign_as_printed, c.reward.w_wait,
      c.reward.w_response, c.reward.w_slo_violation] for name, c in configs.items()],
    caption="**Variants of this notebook** (each retrains the agent; the baselines are unaffected).",
)

## Run every variant

Each job trains one agent for one variant and one seed and evaluates it, together with the
baselines and the SRTF reference, on that seed's paired evaluation workloads
(`experiments.run_scheduler_benchmark.run_seed`, the function the benchmark itself uses). Jobs
run in `WORKERS` processes. The cell checks that the baselines produced identical metrics in
every variant (they must: no variant changes them) and prints any training divergence.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], BASE_OVERRIDES, SEEDS, extra=f"variants: {VARIANTS}; workers: {WORKERS}")
jobs = [(name, {"cfg": configs[name], "bcfg": bcfg, "seed": seed, "skip_ddqn": False, "checkpoint_dir": None})
        for name in VARIANTS for seed in SEEDS]
started = time.perf_counter()
if WORKERS > 1:
    with multiprocessing.get_context("spawn").Pool(processes=min(WORKERS, len(jobs))) as pool:
        results = pool.map(run_seed, [job for _, job in jobs], chunksize=1)
else:
    results = [run_seed(job) for _, job in jobs]
print(f"{len(jobs)} jobs in {time.perf_counter() - started:.0f} s")

rows_by_variant = {name: [] for name in VARIANTS}
for (name, job), result in zip(jobs, results):
    rows_by_variant[name].extend(result["rows"])
    info = result["info"]
    if info.get("diverged"):
        print(f"WARNING {name} seed {job['seed']}: training diverged ({info.get('divergence_reason')})")
summaries = {name: summarise(rows) for name, rows in rows_by_variant.items()}

baseline_key = lambda r: (r["seed"], r["method"], r["load_profile"], r["episode"], r["avg_turnaround_ms"])
reference_baselines = sorted(baseline_key(r) for r in rows_by_variant["default"] if r["method"] != DDQN_METHOD and not r["method"].startswith("ddqn"))
identical = all(
    sorted(baseline_key(r) for r in rows if not r["method"].startswith("ddqn")) == reference_baselines
    for rows in rows_by_variant.values()
)
print(f"baseline and SRTF rows identical across variants: {identical}")
write_csv(OUTPUT_DIR / "raw_episodes_by_variant.csv",
          [{"variant": name, **row} for name, rows in rows_by_variant.items() for row in rows],
          ("variant", *RAW_EPISODE_FIELDS))

In [ ]:
# Helpers shared by the reward and learning-rate sections (no new metric: every number comes
# from summarise / summarise_by_seed, and the comparison with the default uses make_row).
COLUMNS = [
    (BY_KEY["avg_turnaround_ms"], "all", "Turnaround, pooled"),
    (BY_KEY["avg_turnaround_ms"], "heavy", "Turnaround, heavy load"),
    (BY_KEY["avg_response_ms"], "all", "Response time, pooled"),
    (BY_KEY["slo_violation_rate"], "all", "SLO violation rate, pooled"),
    (BY_KEY["starvation_count"], "all", "Starvation count, pooled"),
    (MetricCfg("guardrail_overrides", "Guardrail overrides", "decisions/episode", False, 1e-9, 2, 1.0), "all",
     "Guardrail overrides, pooled"),
]


def per_seed(variant, method, key, load):
    selected = [dict(r, load_profile="all") if load == "all" else r
                for r in rows_by_variant[variant] if r["method"] == method and load in ("all", r["load_profile"])]
    return {row["seed"]: row[f"{key}_mean"] for row in summarise_by_seed(selected, ["seed"], [key])}


def mval(variant, method, metric, load):
    return measurement(find_row(summaries[variant], method=method, load_profile=load), metric)


def variant_table(names, title):
    headline_metric, headline_load, _ = COLUMNS[0]
    default_value = mval("default", AFTER, headline_metric, headline_load)
    body = []
    for name in names:
        cells = [format_mean_std(mval(name, AFTER, m, load).mean, mval(name, AFTER, m, load).std, m.digits)
                 for m, load, _ in COLUMNS]
        seeds = per_seed(name, AFTER, headline_metric.key, headline_load)
        comparison = make_row(headline_metric.spec, "default", default_value, name,
                              mval(name, AFTER, headline_metric, headline_load))
        body.append([f"`{name}`", changes[name], *cells,
                     ", ".join(f"{s}: {v:.1f}" for s, v in seeds.items()),
                     "reference" if name == "default" else row_cells(comparison)["Improvement %"]])
    show_table(
        ["Variant", "What changed", *[f"{label} ({m.unit})" for m, _, label in COLUMNS],
         "Pooled turnaround per seed (ms)", "Pooled turnaround vs default (Improvement %)"],
        body,
        caption=f"**{title} — Double DQN (guardrails on), {RUN_LABEL} {REPRODUCED}.** Mean ± sample std "
                "across seeds. In the last column a positive value means a lower pooled turnaround than the "
                "default; it is computed with `make_row` (sign convention of `src/common/before_after.py`). "
                "Reference ablations: `experiments/results/RESULTS.md`.",
    )


def variant_chart(names, title, filename):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), squeeze=False)
    for ax, (metric, load, label) in zip(axes.ravel(), COLUMNS):
        means = [mval(n, AFTER, metric, load).mean for n in names]
        stds = [mval(n, AFTER, metric, load).std for n in names]
        colors = [plotting.AFTER_COLOR if n == "default" else plotting.REFERENCE_COLOR for n in names]
        bar_panel(ax, names, means, stds, colors, ylabel=f"{metric.label} ({metric.unit})", title=label,
                  digits=metric.digits)
        tops = [ax.get_ylim()[1]]
        for x, name in enumerate(names):
            values = list(per_seed(name, AFTER, metric.key, load).values())
            ax.scatter([x] * len(values), values, s=16, color=plotting.INK, zorder=3,
                       label="one seed" if x == 0 else None)
            tops.append(max(values, default=0) * 1.15)
        if metric.key != "guardrail_overrides":
            for method, style, color in (("sjf", (0, (5, 3)), plotting.BEFORE_COLOR),
                                         ("srtf", (0, (1, 2)), plotting.REFERENCE_COLOR)):
                ref = mval("default", method, metric, load)
                if ref.available:
                    ax.axhline(ref.mean, color=color, linestyle=style, linewidth=1.3, label=LABELS[method])
                    tops.append(ref.mean * 1.15)
        ax.set_ylim(0, max(tops))
    handles, names = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, names, loc="upper right", ncols=len(handles), bbox_to_anchor=(0.995, 0.995), fontsize=8.5)
    fig.suptitle(f"{title} {REPRODUCED} — {RUN_LABEL}", x=0.01, ha="left", fontsize=12, fontweight="bold")
    fig.text(0.01, 0.002, "Orange = default settings, grey = variant; dots = per-seed values; error bars = sample std "
             "across seeds (indicative only with fewer than five seeds).\nDashed blue = SJF, dotted grey = SRTF "
             "reference, both on the same workloads. Synthetic workloads, simulator only.",
             fontsize=8.5, color=plotting.INK_SECONDARY)
    fig.tight_layout(rect=(0, 0.05, 1, 0.93))
    fig.savefig(OUTPUT_DIR / filename, dpi=120)
    plt.show()

## 1. Reward: the sign convention of the response-time term, and two weight variants

- `default` — the reward used here: response time is a penalty, so a faster response earns
  more reward.
- `reward_paper_sign_as_printed` — an alternative sign convention for the response-time term
  (response time is added instead of subtracted), kept as an ablation, so a longer response
  time adds to the reward.
- `reward_turnaround_only` — only the time-in-system penalty (a pure mean-turnaround objective).
- `reward_responsive` — stronger response-time and deadline penalties.

How to read it: look first at how response time and the SLO violation rate differ between the
alternative sign and the reward used here, then at whether the weight variants move the
metrics by more than the seed-to-seed spread (compare the dots, not only the bar heights). A
variant whose dots overlap the default's does not differ in this run.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], BASE_OVERRIDES, SEEDS, extra=f"variants: {['default', *REWARD_VARIANTS]}")
variant_table(["default", *REWARD_VARIANTS], "Reward definition and weights")
variant_chart(["default", *REWARD_VARIANTS], "Sensitivity to the reward", "reward_sensitivity.png")

## 2. Load level: light, medium and heavy

The default agent and every baseline on each load profile separately (same workloads for every
method). The second table shows the per-workload distribution of the turnaround time — median,
90th percentile and maximum over all evaluation workloads of all seeds — and how many workloads
the agent finished with a lower turnaround than SJF and than SRTF (paired, workload by
workload).

How to read it: under light load the methods are close because queues are short; the
differences, and the agent's failures, appear as the load rises. If the agent's median is close
to SJF's but its maximum is far above, the average is being driven by a few bad workloads: a
tail-risk result, not a uniform one.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], BASE_OVERRIDES, SEEDS, extra="variant: default")
LOADS = list(configs["default"].evaluation.load_profiles)
turnaround, slo = BY_KEY["avg_turnaround_ms"], BY_KEY["slo_violation_rate"]
show_table(
    ["Method", "Role", *[f"Turnaround, {load} (ms)" for load in LOADS], *[f"SLO violation rate, {load}" for load in LOADS]],
    [[LABELS[m], ROLE[m]]
     + [format_mean_std(mval("default", m, turnaround, l).mean, mval("default", m, turnaround, l).std, 1) for l in LOADS]
     + [format_mean_std(mval("default", m, slo, l).mean, mval("default", m, slo, l).std, 4) for l in LOADS]
     for m in ORDER],
    caption=f"**Every method by load profile — {RUN_LABEL} {REPRODUCED}.** Mean ± sample std across seeds.",
)

default_rows = rows_by_variant["default"]
by_workload = {(r["seed"], r["load_profile"], r["episode"], r["method"]): float(r["avg_turnaround_ms"]) for r in default_rows}
body = []
for load in LOADS:
    keys = sorted({(s, e) for (s, l, e, _m) in by_workload if l == load})
    for method in ("sjf", "srtf", AFTER):
        values = np.array([by_workload[(s, load, e, method)] for s, e in keys])
        wins_sjf = sum(by_workload[(s, load, e, AFTER)] < by_workload[(s, load, e, "sjf")] for s, e in keys)
        wins_srtf = sum(by_workload[(s, load, e, AFTER)] < by_workload[(s, load, e, "srtf")] for s, e in keys)
        body.append([load, LABELS[method], len(values), f"{np.median(values):.1f}", f"{np.percentile(values, 90):.1f}",
                     f"{values.max():.1f}",
                     f"{wins_sjf}/{len(keys)} vs SJF, {wins_srtf}/{len(keys)} vs SRTF" if method == AFTER else ""])
show_table(["Load", "Method", "Workloads", "Median turnaround (ms)", "90th percentile (ms)", "Maximum (ms)",
            "Agent better on"], body,
           caption=f"**Per-workload turnaround distribution, seeds pooled — {RUN_LABEL} {REPRODUCED}.**")

fig, axes = plt.subplots(2, len(LOADS), figsize=(5.2 * len(LOADS), 7.6), squeeze=False)
colors = [plotting.ROLE_COLORS[ROLE[m]] for m in ORDER]
for row, metric in enumerate((turnaround, slo)):
    for col, load in enumerate(LOADS):
        bar_panel(axes[row][col], ORDER, [mval("default", m, metric, load).mean for m in ORDER],
                  [mval("default", m, metric, load).std for m in ORDER], colors,
                  ylabel=f"{metric.label} ({metric.unit})", title=f"{metric.label}, {load} load", digits=metric.digits)
fig.suptitle(f"Sensitivity to the load level {REPRODUCED} — {RUN_LABEL}", x=0.01, ha="left",
             fontsize=12, fontweight="bold")
fig.text(0.01, 0.002, "Blue = Before baselines, orange = Double DQN with guardrails (default settings), grey = references. "
         "Error bars = sample std across seeds. Synthetic workloads, simulator only.", fontsize=8.5,
         color=plotting.INK_SECONDARY)
fig.tight_layout(rect=(0, 0.03, 1, 0.95))
fig.savefig(OUTPUT_DIR / "load_sensitivity.png", dpi=120)
plt.show()

## 3. One hyperparameter at a time: the learning rate at three values

`lr_low`, `default` and `lr_high` differ only in the Adam learning rate; everything else,
including the training budget and the seeds, is the same.

How to read it: if the three bars differ by more than the spread of the dots, the result
depends on the learning rate at this budget. In the reference run several variants (a lower
learning rate among them) did better than the default on the reporting seeds; the default was
**not** changed afterwards, because choosing the setting that looks best on the reporting seeds
would be tuning on the test set. The same applies to anything this notebook shows.

In [ ]:
provenance([SCHEDULER_CONFIG, BASELINES_CONFIG], BASE_OVERRIDES, SEEDS, extra=f"variants: {['lr_low', 'default', 'lr_high']}")
lr_names = ["lr_low", "default", "lr_high"]
variant_table(lr_names, "Learning rate at three values")
variant_chart(lr_names, "Sensitivity to the learning rate", "learning_rate_sensitivity.png")

## Interpretation and limits

- **Sensitivity, not selection.** Each section changes exactly one thing. The useful question
  is "how much does the result move?", not "which variant wins?". The default settings were
  fixed on development seeds before the reference run; nothing in this notebook changes them,
  and a variant that looks better here is evidence that the result depends on that setting,
  not a recommendation.
- **Notebook scale.** Every variant here uses a reduced training budget (and, if you reduced
  `SEEDS`, fewer seeds) than the reference ablations (both printed above each table). The level of every result differs
  from the reference run; the reference ablation table is in `experiments/results/RESULTS.md`.
  With fewer than five seeds the error bars are only indicative, and because rare catastrophic
  workloads drive the averages, differences between variants on a few seeds may be chance.
- **Alternative reward sign.** Under the alternative sign a longer response time adds to the reward, so
  an agent trained with it is expected to show longer response times or more deadline misses
  than one trained with the reward used here; if the difference is within the spread, this run
  cannot tell them apart. If the
  alternative sign looks *better* on some metric, report that as it is: the response-time term is
  one of several reward terms, and at a reduced budget the other terms can dominate. Check it
  against the reference ablation table before drawing any conclusion. Episode
  returns of different reward variants are not comparable and are not used.
- **Load.** Read the per-workload distribution next to the means: a mean that is worse than
  SJF together with a median that is close to SJF is a tail-risk result. Count the paired
  workload wins against SRTF as well as against SJF: SRTF is the like-for-like preemptive
  reference and is not a Before method.
- **Guardrails.** The override column shows how often the guardrails intervened for each
  variant. Frequent overrides in a variant with poor heavy-load results do not mean the
  guardrails helped: in the reference run they did not contain the failure mode that occurred.
- **Scope.** Single simulated CPU, synthetic workloads, every setting
  `[Assumption — not in paper]`.